# The Projection Path: `.npz` → serialized projection → render

A hands-on decomposition of **one heavy renderer** into three explicit stages, to feel out where
the pieces of an ETL-forward pipeline want to live.

Worked example: **`multi_stream_specialization`** ([renderer](../../../packages/miscope/src/miscope/visualization/renderers/multi_stream_specialization.py)).
It is a near-ideal teaching case because the renderer already splits cleanly:

| current renderer code | nature | belongs to |
|---|---|---|
| `_compute_mlp_band_counts`, `_compute_attn_aggregate`, `_compute_embedding_dim_counts` | deterministic reshape / aggregate over cross-epoch `.npz` | **projection** |
| `render_multi_stream_specialization` (subplots, colors, cursor, layout) | presentation | **renderer (stays)** |

**Terminology.** "View" is already taken in this codebase (the universal View Catalog). Throughout this
notebook the SQL-sense thing — a materialized, tidy, columnar derivation of an artifact — is a **projection**.

**What we are probing.** Not "can we write a parquet file" (we can). We want to find *where the seam falls* —
which transforms are safe to materialize, and what the missing **data-store surface** (serialize + deserialize)
needs to expose so renderers never touch a storage format directly.

In [ ]:
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd

from miscope import load_family

# Canon: fresh artifacts, just passed regression. Reached through the API
# (load_family -> get_variant), never a file path -- the storage-encapsulation
# invariant applies to notebooks too.
family = load_family("modulo_addition_1layer")
variant = family.get_variant(prime=113, seed=999, data_seed=598)
prime = int(variant.model_config["prime"])
print(variant.name, "| prime =", prime, "| n_freq =", prime // 2)

## Stage 0 — the raw store (`.npz`)

The current read surface is `variant.artifacts.*` ([ArtifactLoader](../../../packages/miscope/src/miscope/analysis/artifact_loader.py)):
`load_cross_epoch`, `load_epochs(fields=...)`, `load_summary`. These are honest accessors — but they hand back
**raw numpy** keyed by analyzer-internal names. The consumer still has to know what `dominant_freq` and `max_frac`
*mean* and how to reshape them. That knowledge currently lives in the renderer.

In [ ]:
# Two of the four sources multi_stream draws on (the two with the cleanest tidy form).
neuron_dynamics = variant.artifacts.load_cross_epoch("neuron_dynamics")
eff_dim = variant.artifacts.load_summary("weight_spectra")

epochs = neuron_dynamics["epochs"]
dominant_freq = neuron_dynamics["dominant_freq"]   # (n_epochs, d_mlp)
max_frac = neuron_dynamics["max_frac"]             # (n_epochs, d_mlp)
n_epochs, d_mlp = dominant_freq.shape

print("neuron_dynamics keys :", list(neuron_dynamics.keys()))
print("epochs               :", epochs.shape)
print("dominant_freq        :", dominant_freq.shape, dominant_freq.dtype)
print("max_frac             :", max_frac.shape, max_frac.dtype)
print("in-memory bytes      :", sum(a.nbytes for a in neuron_dynamics.values()))
print("weight_spectra keys  :", [k for k in eff_dim if k.startswith('pr_')][:6], '...')

## Stage 1 — the projection (`.npz` → tidy frame → parquet)

### The seam falls at **threshold-independence**, not at the renderer boundary

This is the central finding. The MLP panel's `_compute_mlp_band_counts` does two things:

1. **reshape** `(epoch, neuron)` arrays into a tidy table — *threshold-independent*
2. **threshold + count** committed neurons per frequency — *depends on `threshold_mlp` (a UI slider)*

If we materialize past step 2, we freeze the slider into storage. So the projection is **step 1 only**: a tidy,
long-form table of the raw per-neuron quantities. The threshold stays a *read-time query* on the projection.

This is not a new opinion — `neuron_dynamics.raw` already made exactly this call ("expose `dominant_freq`/`max_frac`
so consumers can compute per-band specialization at any threshold"). The projection layer is where that principle
gets *serialized* instead of recomputed from `.npz` on every render.

In [ ]:
# Projection A: MLP per-neuron specialization -- tidy, long, threshold-INDEPENDENT.
mlp_long = pd.DataFrame(
    {
        "epoch": np.repeat(epochs, d_mlp),
        "neuron": np.tile(np.arange(d_mlp), n_epochs),
        "dominant_freq": dominant_freq.reshape(-1),
        "max_frac": max_frac.reshape(-1),
    }
)

# Projection B: effective dimensionality -- already scalar-per-(epoch, matrix), trivially tidy.
ed_epochs = eff_dim["epochs"]
rows = []
for matrix in ["W_E", "W_in", "W_out", "W_O"]:
    key = f"pr_{matrix}"
    if key not in eff_dim:
        continue
    arr = eff_dim[key]
    vals = arr.mean(axis=1) if arr.ndim == 2 else arr
    rows.append(pd.DataFrame({"epoch": ed_epochs, "matrix": matrix, "participation_ratio": vals}))
eff_long = pd.concat(rows, ignore_index=True)

print("mlp_long :", mlp_long.shape, "->", list(mlp_long.columns))
print("eff_long :", eff_long.shape, "->", list(eff_long.columns))
mlp_long.head()

In [ ]:
# Serialize. NOTE: in a real Store this path is composed BY the Store, not the consumer.
# We hand-roll it here precisely to expose what the Store should encapsulate.
scratch = Path("_projection_scratch")
scratch.mkdir(exist_ok=True)
mlp_path = scratch / "mlp_specialization_raw.parquet"
eff_path = scratch / "effective_dimensionality.parquet"
mlp_long.to_parquet(mlp_path, index=False)
eff_long.to_parquet(eff_path, index=False)
print("wrote", mlp_path, mlp_path.stat().st_size, "bytes")
print("wrote", eff_path, eff_path.stat().st_size, "bytes")

### An honest wrinkle: parquet size is *not* the win here

`mlp_specialization_raw.parquet` is roughly the same size as the source `.npz` slice — sometimes larger. Long-form
**duplicates the `epoch`/`neuron` index columns**, and `.npz` is already compressed. The starting-frame assumption
"lower memory footprint" does *not* hold uniformly. Where the projection layer actually pays:

- **No recompute.** The embedding panel (`_compute_embedding_dim_counts`) rebuilds a full Fourier projection of
  `W_E` *on every render*, and loads the whole `parameter_snapshot` `W_E` stack to do it. Materializing that once is
  the real memory + latency win — far more than the per-neuron table shown here.
- **Columnar / predicate pushdown.** Read only the columns and epoch-rows you need, instead of `np.load`-ing the
  full array. This matters most on the *large* cross-epoch artifacts.
- **Queryability.** The threshold-at-read step below is a `groupby`, not a bespoke numpy massage buried in a renderer.

So the payoff concentrates on **heavy-compute, multi-source, trajectory** panels — exactly where we said to aim.

In [ ]:
# The threshold lives HERE -- a read-time query on the projection, not baked into storage.
def mlp_band_counts(projection: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """committed neurons per (epoch, dominant_freq) at an arbitrary threshold."""
    committed = projection[projection["max_frac"] >= threshold]
    return committed.groupby(["epoch", "dominant_freq"]).size().reset_index(name="n")

t0 = time.time()
back = pd.read_parquet(mlp_path)                 # deserialize
band_counts = mlp_band_counts(back, threshold=0.7)
print(f"read + threshold(0.7) + groupby: {(time.time() - t0) * 1000:.1f} ms")
print("band rows:", len(band_counts))
# Same projection, a different slider value -- no .npz reload, no recompute.
print("at threshold 0.5 :", len(mlp_band_counts(back, 0.5)), "band rows")
band_counts.head()

## Stage 2 — render from the projection

The renderer now consumes **tidy frames** and does only presentation. Compare against the original
[`render_multi_stream_specialization`](../../../packages/miscope/src/miscope/visualization/renderers/multi_stream_specialization.py#L111):
the `_compute_*` helpers are gone from render time — they moved upstream into the projection. What's left is
subplots, colors, cursor, layout (this is the part that *stays* in the renderer).

In [ ]:
import colorsys

import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _freq_color(k: int, n_freq: int) -> str:
    r, g, b = colorsys.hls_to_rgb((k - 1) / n_freq, 0.55, 0.5)
    return f"rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})"


def render_from_projection(band_counts, eff_long, prime, epoch=None):
    """Pure presentation -- input is already-tidy projection frames."""
    n_freq = prime // 2
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
        subplot_titles=[
            "MLP \u2014 committed neurons per frequency (threshold applied at read)",
            "Effective dimensionality (participation ratio)",
        ],
    )
    pivot = band_counts.pivot_table(index="epoch", columns="dominant_freq", values="n", fill_value=0)
    for k in pivot.columns:
        fig.add_trace(
            go.Scatter(x=pivot.index, y=pivot[k], mode="lines", name=f"Freq {int(k) + 1}",
                       line=dict(color=_freq_color(int(k) + 1, n_freq), width=1.5)),
            row=1, col=1,
        )
    for matrix, sub in eff_long.groupby("matrix"):
        fig.add_trace(
            go.Scatter(x=sub["epoch"], y=sub["participation_ratio"], mode="lines", name=matrix,
                       line=dict(width=2)),
            row=2, col=1,
        )
    if epoch is not None:
        fig.add_vline(x=epoch, line_color="rgba(180,0,0,0.6)", line_width=1.5)
    fig.update_layout(template="plotly_white", height=720, width=900, hovermode="x unified",
                      title=f"Multi-stream (from projection) \u2014 p={prime}")
    fig.update_yaxes(title_text="Neuron count", row=1, col=1)
    fig.update_yaxes(title_text="Participation ratio", row=2, col=1)
    fig.update_xaxes(title_text="Epoch", row=2, col=1)
    return fig


render_from_projection(band_counts, eff_long, prime, epoch=int(epochs[len(epochs) // 2]))

## Where the pieces belong

What this slice argues, concretely:

**1. A projection is a DAG node, not a new phase.** Strip the rendering away and `mlp_long` is produced by a node
whose inputs are `ArtifactInput("neuron_dynamics")` (+ others), with no `ModelInput` — structurally identical to a
REQ_133 *former-secondary* analyzer. It rides the existing topo-sort, two-pass execution, and (critically) the
transitive-staleness machinery for free. The only genuinely new thing is its **output encoding**: parquet, not `.npz`.

**2. The missing abstraction is a Store, and the verb falls out of it.** Notice every IO line above reaches a
*format* directly: `variant.artifacts.load_cross_epoch` returns raw numpy; `pd.to_parquet` / `pd.read_parquet`
compose paths by hand. A **data-store surface** would own both halves — serialize (write the projection) and
deserialize (return a frame against a code-declared schema) — so a renderer asks for *a named projection* and never
names a format or a path. The `load_table`-style verb is then just the Store's read method; it isn't the abstraction,
it's a consequence of it.

**3. The seam is threshold-independence.** Materialize the deterministic reshape; keep parameterized steps
(thresholds, sort kwargs) as read-time queries. Cross this line and you bake the UI into storage.

**4. Existing precedent, deliberately not leaned on.** `DataView` / `DataViewCatalog` (REQ_054) is the *read* half of
this idea — a schema-bearing surface returning frames — but it is currently **undeveloped** and still loads `.npz` on
demand (no serialize half). It's a precedent for the *shape* of the Store, not a foundation to build on as-is.

**Open question this slice sharpened (for you, not to answer here):** is a projection *its own node type* registered
with the planner, or an *output-encoding flag* on the existing analyzer Spec (`output_encoding: npz | parquet`) plus a
Store connector/loader pair? The hand-rolled IO above is exactly the surface that flag-or-node decision would absorb.

In [ ]:
# tidy up scratch artifacts (comment out to inspect the parquet files)
import shutil

shutil.rmtree(scratch, ignore_errors=True)
print("removed", scratch)